# Wishart Graph Dictionary — 100k Colab run

This notebook runs the current `feature/wishart-graph-dictionary` experiment on Google Colab. It clones the exact branch, computes on local `/content` scratch, checkpoints every completed level/transition to Google Drive, and validates the final graph-dictionary artifacts.

**Scientific config:** `configs/wishart_conceptnet_dictionary.yaml` (100k nodes, exact GraphTypes, full frequency scan, frequency-weighted Wishart, MDL selection, recursive grammar, Huffman artifacts).


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys

BRANCH = 'feature/wishart-graph-dictionary'
REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
REPO_DIR = Path('/content/semgraphex')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print({'branch': BRANCH, 'commit': head, 'repo': str(REPO_DIR)})


In [ ]:
# Install the exact checked-out branch and Wishart/notebook dependencies.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', 'requirements/constraints.txt',
    '-e', '.[wishart,notebook]'
], check=True)
print('semmap-haken installed from', head)


In [ ]:
# Google OAuth confirmation is the only interactive step.
from google.colab import drive
drive.mount('/content/drive')


## Source and run selection

The notebook prefers the newest completed prepared graph under `prepared/*/COMPLETED`. If none exists, it falls back to `data/conceptnet_en_100k.tsv`. Set an override only if your Drive layout differs.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/SemanticMap/semgraphex')
CONFIG = REPO_DIR / 'configs/wishart_conceptnet_dictionary.yaml'

# Optional explicit overrides. Leave None for automatic discovery.
PREPARED_OVERRIDE = None  # e.g. 'prepared/prepare-20260923'
DATASET_OVERRIDE = None   # e.g. 'data/conceptnet_en_100k.tsv'

prepared_path = None
dataset_path = None

if PREPARED_OVERRIDE is not None:
    prepared_path = DRIVE_ROOT / PREPARED_OVERRIDE
elif DATASET_OVERRIDE is not None:
    dataset_path = DRIVE_ROOT / DATASET_OVERRIDE
else:
    prepared_root = DRIVE_ROOT / 'prepared'
    completed = list(prepared_root.glob('*/COMPLETED')) if prepared_root.exists() else []
    if completed:
        completed.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        prepared_path = completed[0].parent
    else:
        candidate = DRIVE_ROOT / 'data' / 'conceptnet_en_100k.tsv'
        if candidate.is_file():
            dataset_path = candidate

if prepared_path is None and dataset_path is None:
    raise FileNotFoundError(
        f'No completed prepared graph under {DRIVE_ROOT / "prepared"} and no '
        f'{DRIVE_ROOT / "data" / "conceptnet_en_100k.tsv"}. '
        'Upload one of them or set PREPARED_OVERRIDE/DATASET_OVERRIDE.'
    )

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_NAME = f'wishart-dictionary-100k-{stamp}'
DRIVE_RUN = DRIVE_ROOT / 'runs' / RUN_NAME

print({
    'config': str(CONFIG),
    'prepared': str(prepared_path) if prepared_path else None,
    'dataset': str(dataset_path) if dataset_path else None,
    'run_name': RUN_NAME,
    'drive_output': str(DRIVE_RUN),
})


In [ ]:
# Run the current 100k dictionary experiment. Compute is local; checkpoints are durable on Drive.
command = [
    'semmap-wishart-colab',
    '--config', str(CONFIG),
    '--drive-root', str(DRIVE_ROOT),
    '--scratch-root', '/content/semmap-wishart',
    '--run-name', RUN_NAME,
    '--compare-mode', 'size_mtime',
    '--blas-threads', '1',
]
if prepared_path is not None:
    command += ['--prepared-drive', str(prepared_path)]
else:
    command += ['--dataset-drive', str(dataset_path)]

print('RUN:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
# Validate that this was a graph-dictionary run, not merely a legacy coarsening run.
required = [
    DRIVE_RUN / 'COMPLETED',
    DRIVE_RUN / 'hierarchy.json',
    DRIVE_RUN / 'dictionary' / 'graph_types.jsonl',
    DRIVE_RUN / 'dictionary' / 'grammar.jsonl',
    DRIVE_RUN / 'dictionary' / 'huffman.json',
    DRIVE_RUN / 'dictionary' / 'statistics.json',
    DRIVE_RUN / 'level_000' / 'symbolic_nodes.jsonl',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError({'missing_dictionary_artifacts': missing})

hierarchy = json.loads((DRIVE_RUN / 'hierarchy.json').read_text(encoding='utf-8'))
stats = json.loads((DRIVE_RUN / 'dictionary' / 'statistics.json').read_text(encoding='utf-8'))
print('COMPLETED')
print(json.dumps({
    'initial_nodes': hierarchy['initial_nodes'],
    'final_nodes': hierarchy['final_nodes'],
    'levels': hierarchy['levels'],
    'stop_reason': hierarchy['stop_reason'],
    'dictionary_size': stats['dictionary_size'],
    'candidate_occurrences': stats['candidate_occurrences'],
    'accepted_occurrences': stats['accepted_occurrences'],
    'recursive_types': stats['recursive_types'],
}, indent=2))


In [ ]:
# Compact transition table for immediate experiment inspection.
import pandas as pd

rows = []
for tr in hierarchy.get('transitions', []):
    dm = tr.get('dictionary_metrics', {})
    rows.append({
        'level': tr.get('source_level'),
        'fine_nodes': tr.get('fine_nodes'),
        'coarse_nodes': tr.get('coarse_nodes'),
        'node_compression': tr.get('compression_ratio'),
        'dictionary_size': dm.get('dictionary_size_after_discovery'),
        'new_types': dm.get('new_types'),
        'reused_types': dm.get('reused_types'),
        'novelty': dm.get('dictionary_novelty'),
        'reuse_rate': dm.get('reuse_rate'),
        'recursive_types': dm.get('recursive_types_total'),
        'selected_occurrences': dm.get('selected_occurrences'),
        'mdl_gain_bits_proxy': dm.get('mdl_gain_bits_proxy'),
        'mdl_ratio_proxy': dm.get('mdl_ratio_proxy'),
        'entropy_bits': dm.get('type_entropy_bits'),
        'mean_huffman_bits': dm.get('mean_huffman_code_length'),
    })

table = pd.DataFrame(rows)
display(table)


In [ ]:
# Plot the four trajectories that matter most for the dictionary hypothesis.
import matplotlib.pyplot as plt

if not table.empty:
    for column in ['mdl_ratio_proxy', 'reuse_rate', 'novelty', 'recursive_types']:
        if column not in table or table[column].dropna().empty:
            continue
        plt.figure(figsize=(7, 4))
        plt.plot(table['level'], table[column], marker='o')
        plt.xlabel('compression level')
        plt.ylabel(column)
        plt.title(column)
        plt.grid(True, alpha=0.25)
        plt.show()


## Interpretation gate

The current version uses a **structural MDL proxy**, not measured bytes of a finalized lossless binary codec. A strong dictionary result is therefore a joint trajectory: MDL proxy improves, reuse rises, novelty falls, and recursive composition rises. The durable run is under `MyDrive/SemanticMap/semgraphex/runs/<RUN_NAME>`.
